In [ ]:
!apt-get install -y libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1
!pip install -q fastapi "uvicorn[standard]" python-multipart websockets nest_asyncio
!pip install -q "opencv-python-headless==4.10.0.84" Pillow numpy
!pip install -q "transformers>=4.41.0" torch torchvision accelerate
!pip install -q sentencepiece protobuf qwen-vl-utils
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 17.4 MB/s eta 0:00:00
✅ All packages installed


In [ ]:
ocr_engine_code = '''
import cv2, numpy as np, torch, re, time, warnings
from PIL import Image
warnings.filterwarnings("ignore")

MODEL = None
PROCESSOR = None
DEVICE_INFO = "unknown"

def load_model():
    global MODEL, PROCESSOR, DEVICE_INFO
    if MODEL is not None:
        return MODEL, PROCESSOR
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

    if not torch.cuda.is_available():
        print("⚠️  WARNING: No GPU found! Running on CPU will be VERY slow (60–120s per image).")
        print("   Go to: Runtime → Change runtime type → T4 GPU, then restart and rerun all cells.")
        DTYPE = torch.float32
    else:
        print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
        DTYPE = torch.float16

    print("🔄 Loading Qwen2-VL-2B-Instruct…")
    MODEL = Qwen2VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct",
        torch_dtype=DTYPE,
        device_map="auto",
    )
    PROCESSOR = AutoProcessor.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct",
        min_pixels=256*28*28,
        max_pixels=1280*28*28,
    )
    DEVICE_INFO = str(next(MODEL.parameters()).device)
    print(f"✅ Qwen2-VL ready on {DEVICE_INFO}!")
    return MODEL, PROCESSOR


class Preprocessor:
    def load_bytes(self, image_bytes):
        arr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError("Cannot decode image")
        return img

    def upscale(self, img, min_dim=1200):
        h, w = img.shape[:2]
        if min(h, w) < min_dim:
            s = min_dim / min(h, w)
            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_CUBIC)
        return img

    def remove_shadows(self, img):
        planes = []
        for p in cv2.split(img):
            bg = cv2.medianBlur(cv2.dilate(p, np.ones((7,7), np.uint8)), 21)
            planes.append(cv2.normalize(255 - cv2.absdiff(p, bg), None, 0, 255, cv2.NORM_MINMAX))
        return cv2.merge(planes)

    def auto_rotate(self, img):
        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(cv2.GaussianBlur(gray,(5,5),0), 50, 150, apertureSize=3)
        lines = cv2.HoughLines(edges, 1, np.pi/180, threshold=80)
        if lines is None: return img, 0.0
        angles = [float(np.degrees(l[0][1])-90) for l in lines[:60]
                  if -45 < float(np.degrees(l[0][1])-90) < 45]
        if not angles: return img, 0.0
        angle = float(np.median(angles))
        if abs(angle) < 0.5: return img, 0.0
        h, w = img.shape[:2]; cx, cy = w//2, h//2
        M = cv2.getRotationMatrix2D((cx,cy), angle, 1.0)
        cos, sin = abs(M[0,0]), abs(M[0,1])
        nw, nh = int(h*sin+w*cos), int(h*cos+w*sin)
        M[0,2] += nw/2-cx; M[1,2] += nh/2-cy
        rotated = cv2.warpAffine(img, M, (nw,nh), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        return rotated, angle

    def enhance(self, img):
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
        img = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
        img = cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
        return img

    def denoise(self, img):
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    def run_full(self, img):
        img = self.upscale(img)
        img = self.remove_shadows(img)
        img, angle = self.auto_rotate(img)
        img = self.denoise(img)
        img = self.enhance(img)
        return img, angle

    def run_fast(self, img, max_dim=640):
        """Lightweight preprocess for camera frames."""
        h, w = img.shape[:2]
        if max(h, w) > max_dim:
            s = max_dim / max(h, w)
            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
        return img, 0.0


class DocumentDetector:
    def _color_crop(self, img):
        hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, np.array([0,0,150]), np.array([180,80,255]))
        k    = np.ones((15,15), np.uint8)
        mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: return None
        largest = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(largest) < img.shape[0]*img.shape[1]*0.10: return None
        x,y,w,h = cv2.boundingRect(largest); pad=15
        x=max(0,x-pad); y=max(0,y-pad)
        w=min(img.shape[1]-x,w+2*pad); h=min(img.shape[0]-y,h+2*pad)
        return img[y:y+h, x:x+w]

    def _edge_crop(self, img):
        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.dilate(cv2.Canny(cv2.GaussianBlur(gray,(5,5),0),30,100),
                           np.ones((3,3),np.uint8), iterations=2)
        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        area = img.shape[0]*img.shape[1]
        for cnt in sorted(cnts, key=cv2.contourArea, reverse=True)[:8]:
            peri   = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02*peri, True)
            if len(approx)==4 and cv2.contourArea(cnt)>area*0.15:
                x,y,w,h = cv2.boundingRect(cnt); pad=15
                x=max(0,x-pad); y=max(0,y-pad)
                w=min(img.shape[1]-x,w+2*pad); h=min(img.shape[0]-y,h+2*pad)
                return img[y:y+h, x:x+w]
        return None

    def extract(self, img):
        crop = self._color_crop(img)
        if crop is not None: return crop
        crop = self._edge_crop(img)
        if crop is not None: return crop
        return img


PROMPTS = [
    ("Transcribe every word and number in this image exactly as written. "
     "Preserve line breaks. Output only the text, no explanations."),
    ("Read all text in this image. If there are two columns, format as label then value. "
     "Output only the extracted text."),
    ("List all text items visible in this image. "
     "One item per line. Include all numbers and symbols exactly as shown."),
]

CAMERA_PROMPT = (
    "Transcribe every word, number, and symbol you can see in this image. "
    "Output ONLY the extracted text, one item per line. "
    "If nothing is readable, output: [No text found]"
)


def run_vlm(img_bgr: np.ndarray, prompt: str) -> str:
    model, processor = load_model()
    pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    messages = [{"role":"user","content":[
        {"type":"image","image":pil_img},
        {"type":"text","text":prompt}
    ]}]
    text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = processor(text=[text_in], images=[pil_img], padding=True,
                        return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0].strip()


def _score(text: str) -> int:
    if not text: return 0
    lines      = [l for l in text.splitlines() if l.strip()]
    real_words = [w for w in text.split() if len(w)>1 and sum(c.isalpha() for c in w)/len(w)>0.5]
    euro_hits  = len(re.findall(r"\\d+\\s*[€eE£$]", text))
    garbage    = sum(1 for c in text if not c.isalnum() and c not in " \\n\\t.,;:!?€$£%-_()\\'\\"-/")
    unique_ln  = len(set(lines))
    repeats    = (len(lines)-unique_ln)*20
    return len(real_words)*4 + euro_hits*10 + unique_ln*3 - int(garbage/max(len(text),1)*150) - repeats


preprocessor = Preprocessor()
detector     = DocumentDetector()


def ocr_image_bytes(image_bytes: bytes, auto_crop: bool = True) -> dict:
    t0  = time.time()
    raw = preprocessor.load_bytes(image_bytes)
    if auto_crop:
        raw = detector.extract(raw)
    img, angle = preprocessor.run_full(raw)

    best_text, best_score = "", -999
    for prompt in PROMPTS:
        text = run_vlm(img, prompt)
        s    = _score(text)
        if s > best_score:
            best_text, best_score = text, s

    return {
        "text":            best_text or "[No text detected]",
        "score":           best_score,
        "angle_corrected": round(angle, 2),
        "elapsed":         round(time.time()-t0, 2),
        "device":          DEVICE_INFO,
    }


def ocr_camera_frame(image_bytes: bytes) -> dict:
    """Single fast prompt for camera — skips heavy preprocessing."""
    t0  = time.time()
    raw = preprocessor.load_bytes(image_bytes)
    img, _ = preprocessor.run_fast(raw, max_dim=640)
    text = run_vlm(img, CAMERA_PROMPT)
    return {
        "text":    text or "[No text found]",
        "elapsed": round(time.time()-t0, 2),
        "device":  DEVICE_INFO,
    }
'''

with open("ocr_engine.py", "w") as f:
    f.write(ocr_engine_code.strip())
print("✅ ocr_engine.py written")

✅ ocr_engine.py written


In [ ]:
import pathlib

main_code = '''
import asyncio, base64, json, os, time
from contextlib import asynccontextmanager
from pathlib import Path
from typing import List
from concurrent.futures import ThreadPoolExecutor

import uvicorn
from fastapi import FastAPI, File, Form, UploadFile, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, JSONResponse

executor = ThreadPoolExecutor(max_workers=2)

@asynccontextmanager
async def lifespan(app: FastAPI):
    loop = asyncio.get_event_loop()
    await loop.run_in_executor(executor, _preload_model)
    yield

def _preload_model():
    try:
        from ocr_engine import load_model
        load_model()
    except Exception as e:
        print(f"⚠️  Model preload failed: {e}")

app = FastAPI(title="OCR API", version="1.0.0", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

STATIC_DIR = Path(__file__).parent / "static"
STATIC_DIR.mkdir(exist_ok=True)

@app.get("/", response_class=HTMLResponse)
async def serve_ui():
    html_path = STATIC_DIR / "index.html"
    if html_path.exists():
        return HTMLResponse(content=html_path.read_text(encoding="utf-8"))
    return HTMLResponse(content="<h2>UI not found</h2>")

@app.get("/health")
async def health():
    import torch
    from ocr_engine import DEVICE_INFO
    return {
        "status": "ok",
        "gpu":    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)",
        "device": DEVICE_INFO,
        "timestamp": time.time(),
    }

@app.post("/ocr/image")
async def ocr_single_image(file: UploadFile = File(...), auto_crop: bool = Form(True)):
    from ocr_engine import ocr_image_bytes
    image_bytes = await file.read()
    loop   = asyncio.get_event_loop()
    result = await loop.run_in_executor(executor, ocr_image_bytes, image_bytes, auto_crop)
    result["filename"] = file.filename
    return JSONResponse(content=result)

@app.post("/ocr/batch")
async def ocr_batch_images(files: List[UploadFile] = File(...), auto_crop: bool = Form(True)):
    from ocr_engine import ocr_image_bytes
    loop    = asyncio.get_event_loop()
    results = []
    for f in files:
        image_bytes = await f.read()
        result = await loop.run_in_executor(executor, ocr_image_bytes, image_bytes, auto_crop)
        result["filename"] = f.filename
        results.append(result)
    return JSONResponse(content={"results": results, "count": len(results)})

@app.websocket("/ocr/camera")
async def ocr_camera_ws(websocket: WebSocket):
    from ocr_engine import ocr_camera_frame
    await websocket.accept()
    loop     = asyncio.get_event_loop()
    frame_id = 0
    processing = False   # prevent stacking inference calls

    try:
        while True:
            try:
                raw = await asyncio.wait_for(websocket.receive_text(), timeout=60.0)
            except asyncio.TimeoutError:
                await websocket.send_json({"heartbeat": True})
                continue

            if processing:
                # Inform UI we are still busy with previous frame
                await websocket.send_json({
                    "text":     "⏳ Still processing previous frame…",
                    "elapsed":  0,
                    "frame_id": frame_id,
                    "busy":     True,
                })
                continue

            data = json.loads(raw)
            b64  = data.get("frame", "")
            if not b64:
                await websocket.send_json({"error": "empty frame"})
                continue

            if "," in b64:
                b64 = b64.split(",", 1)[1]
            image_bytes = base64.b64decode(b64)
            frame_id   += 1
            processing  = True

            try:
                result = await loop.run_in_executor(executor, ocr_camera_frame, image_bytes)
                result["frame_id"] = frame_id
                result["busy"]     = False
                await websocket.send_json(result)
            finally:
                processing = False

    except WebSocketDisconnect:
        print("📷 Camera WS disconnected")
    except Exception as e:
        print(f"⚠️  Camera WS error: {e}")
        try: await websocket.send_json({"error": str(e)})
        except: pass

if __name__ == "__main__":
    PORT = int(os.environ.get("PORT", 8000))
    uvicorn.run("main:app", host="0.0.0.0", port=PORT, reload=False, log_level="info")
'''

pathlib.Path("main.py").write_text(main_code.strip())
print("✅ main.py written")

✅ main.py written


In [ ]:
import pathlib

html_code = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>OCR Studio — Qwen2-VL</title>
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#0d1117;--surface:#161b22;--surface2:#21262d;--border:#30363d;
  --accent:#58a6ff;--accent2:#7ee787;--text:#e6edf3;--muted:#8b949e;
  --red:#f85149;--warn:#f0883e;--radius:12px;--shadow:0 8px 32px rgba(0,0,0,.6);
}
html,body{height:100%;background:var(--bg);color:var(--text);font-family:'Segoe UI',system-ui,sans-serif;font-size:15px}
.app{display:flex;flex-direction:column;min-height:100vh}
header{background:var(--surface);border-bottom:1px solid var(--border);padding:16px 28px;display:flex;align-items:center;gap:14px;flex-wrap:wrap}
header h1{font-size:1.3rem;font-weight:700;color:var(--accent)}
.badge{border-radius:20px;padding:3px 12px;font-size:.75rem;border:1px solid var(--border);background:var(--surface2);color:var(--accent2)}
.badge.warn{border-color:var(--warn);color:var(--warn)}
.badge.err{border-color:var(--red);color:var(--red)}
main{flex:1;max-width:1200px;width:100%;margin:0 auto;padding:24px 20px;display:flex;flex-direction:column;gap:20px}
.warn-banner{background:rgba(240,136,62,.12);border:1px solid var(--warn);border-radius:10px;padding:14px 18px;color:var(--warn);font-size:.9rem;display:none}
.warn-banner.show{display:block}
.tabs{display:flex;gap:8px}
.tab{padding:9px 24px;border-radius:8px;border:1px solid var(--border);background:var(--surface);cursor:pointer;font-size:.9rem;color:var(--muted);transition:all .2s;user-select:none}
.tab:hover{border-color:var(--accent);color:var(--text)}
.tab.active{background:var(--accent);border-color:var(--accent);color:#000;font-weight:600}
.card{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:24px;box-shadow:var(--shadow)}
.card-title{font-size:.95rem;font-weight:600;color:var(--accent);margin-bottom:18px;display:flex;align-items:center;gap:8px}
.panel{display:grid;grid-template-columns:1fr 1fr;gap:20px}
@media(max-width:720px){.panel{grid-template-columns:1fr}}
.dropzone{border:2px dashed var(--border);border-radius:var(--radius);background:var(--surface2);min-height:200px;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;cursor:pointer;transition:all .2s;padding:20px;text-align:center}
.dropzone:hover,.dropzone.drag-over{border-color:var(--accent);background:rgba(88,166,255,.05)}
.dropzone svg{width:44px;height:44px;opacity:.5}
.dropzone p{color:var(--muted);font-size:.88rem}
.dropzone strong{color:var(--text)}
#file-input{display:none}
.preview-grid{display:flex;flex-wrap:wrap;gap:8px;margin-top:14px}
.preview-item{position:relative;width:80px;height:80px;border-radius:8px;overflow:hidden;border:1px solid var(--border)}
.preview-item img{width:100%;height:100%;object-fit:cover}
.preview-item .remove{position:absolute;top:2px;right:2px;background:var(--red);border:none;color:#fff;border-radius:50%;width:18px;height:18px;cursor:pointer;font-size:11px;line-height:18px;text-align:center;display:none}
.preview-item:hover .remove{display:block}
.controls{display:flex;flex-direction:column;gap:14px;margin-top:18px}
.control-row{display:flex;align-items:center;gap:10px;flex-wrap:wrap}
label.toggle{display:flex;align-items:center;gap:8px;cursor:pointer;color:var(--muted);font-size:.88rem}
input[type=checkbox]{width:15px;height:15px;accent-color:var(--accent);cursor:pointer}
.btn{padding:9px 22px;border-radius:8px;border:none;cursor:pointer;font-size:.9rem;font-weight:600;transition:all .18s}
.btn-primary{background:var(--accent);color:#000}
.btn-primary:hover:not(:disabled){background:#79b8ff}
.btn-primary:disabled{opacity:.4;cursor:not-allowed}
.btn-danger{background:var(--red);color:#fff}
.btn-danger:hover{background:#da3633}
.btn-ghost{background:var(--surface2);color:var(--text);border:1px solid var(--border)}
.btn-ghost:hover{border-color:var(--accent)}
.output-box{background:var(--surface2);border:1px solid var(--border);border-radius:var(--radius);padding:18px;min-height:200px;max-height:460px;overflow-y:auto;font-family:'Courier New',monospace;font-size:.86rem;line-height:1.9;color:var(--accent2);white-space:pre-wrap;word-break:break-word}
.output-box.empty{color:var(--muted);font-style:italic;font-family:inherit}
.output-box.busy{color:var(--warn);font-style:italic;font-family:inherit}
.output-meta{display:flex;gap:12px;margin-top:10px;flex-wrap:wrap}
.meta-chip{background:var(--surface2);border:1px solid var(--border);border-radius:20px;padding:2px 10px;font-size:.76rem;color:var(--muted)}
.meta-chip span{color:var(--text);font-weight:600}
.batch-result{border-bottom:1px solid var(--border);padding:14px 0}
.batch-result:last-child{border-bottom:none}
.batch-filename{font-size:.83rem;color:var(--accent);font-weight:600;margin-bottom:8px;display:flex;justify-content:space-between}
.batch-text{font-family:'Courier New',monospace;font-size:.83rem;color:var(--accent2);white-space:pre-wrap;line-height:1.7}
.copy-btn{padding:4px 12px;font-size:.78rem;border-radius:6px;background:var(--surface2);border:1px solid var(--border);color:var(--muted);cursor:pointer;transition:all .15s}
.copy-btn:hover{color:var(--text);border-color:var(--accent)}
.copy-btn.copied{color:var(--accent2);border-color:var(--accent2)}
.status{display:flex;align-items:center;gap:8px;font-size:.88rem;color:var(--muted);min-height:26px;margin-top:10px}
.spinner{width:16px;height:16px;border:2px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:spin .8s linear infinite;flex-shrink:0}
@keyframes spin{to{transform:rotate(360deg)}}
.cam-layout{display:grid;grid-template-columns:1fr 1fr;gap:20px}
@media(max-width:720px){.cam-layout{grid-template-columns:1fr}}
#camera-video{width:100%;border-radius:10px;border:2px solid var(--border);background:#000;aspect-ratio:4/3;object-fit:cover}
#camera-canvas{display:none}
.live-badge{display:inline-flex;align-items:center;gap:6px;background:rgba(240,80,80,.15);border:1px solid var(--red);border-radius:20px;padding:3px 12px;font-size:.76rem;color:var(--red);font-weight:600}
.live-dot{width:7px;height:7px;background:var(--red);border-radius:50%;animation:blink 1s infinite}
@keyframes blink{0%,100%{opacity:1}50%{opacity:.2}}
.cam-controls{display:flex;align-items:center;gap:10px;flex-wrap:wrap;margin-bottom:14px}
.frame-info{color:var(--muted);font-size:.8rem}
.section{display:none}
.section.active{display:flex;flex-direction:column;gap:18px}
.mode-pills{display:flex;gap:6px}
.pill{padding:5px 16px;border-radius:20px;border:1px solid var(--border);background:var(--surface2);cursor:pointer;font-size:.83rem;color:var(--muted);transition:all .18s}
.pill.active{border-color:var(--accent);color:var(--accent);background:rgba(88,166,255,.1)}
#toast{position:fixed;bottom:24px;left:50%;transform:translateX(-50%);background:var(--accent2);color:#000;padding:9px 22px;border-radius:8px;font-weight:600;opacity:0;transition:opacity .3s;pointer-events:none;z-index:9999}
#toast.show{opacity:1}
progress{width:100%;height:6px;border-radius:3px;margin-top:8px;accent-color:var(--accent)}
</style>
</head>
<body>
<div class="app">
  <header>
    <svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#58a6ff" stroke-width="2">
      <rect x="3" y="3" width="18" height="18" rx="2"/>
      <path d="M7 8h10M7 12h10M7 16h6"/>
    </svg>
    <h1>OCR Studio</h1>
    <span style="color:var(--muted);font-size:.85rem">Qwen2-VL-2B</span>
    <div style="flex:1"></div>
    <span class="badge" id="gpu-badge">Checking…</span>
  </header>

  <main>
    <!-- GPU warning banner -->
    <div class="warn-banner" id="cpu-warn">
      ⚠️ <strong>No GPU detected — running on CPU.</strong>
      Each image takes 60–120 seconds. Go to <strong>Runtime → Change runtime type → T4 GPU</strong> then restart all cells for best performance.
    </div>

    <div class="tabs">
      <div class="tab active" data-tab="image" onclick="switchTab('image')">🖼️ Image OCR</div>
      <div class="tab" data-tab="camera" onclick="switchTab('camera')">📷 Live Camera OCR</div>
    </div>

    <!-- ═══ IMAGE SECTION ═══ -->
    <div class="section active" id="sec-image">
      <div style="display:flex;align-items:center;gap:14px">
        <span style="color:var(--muted);font-size:.88rem">Mode:</span>
        <div class="mode-pills">
          <div class="pill active" id="pill-single" onclick="setImageMode('single')">Single Image</div>
          <div class="pill" id="pill-batch" onclick="setImageMode('batch')">Batch (Multiple)</div>
        </div>
      </div>

      <div class="panel">
        <div class="card">
          <div class="card-title">
            <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
              <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"/>
              <polyline points="17 8 12 3 7 8"/><line x1="12" y1="3" x2="12" y2="15"/>
            </svg>
            Upload
          </div>
          <div class="dropzone" id="dropzone"
               onclick="document.getElementById('file-input').click()"
               ondragover="onDragOver(event)" ondragleave="onDragLeave(event)" ondrop="onDrop(event)">
            <svg viewBox="0 0 24 24" fill="none" stroke="#58a6ff" stroke-width="1.5">
              <rect x="3" y="3" width="18" height="18" rx="2"/>
              <path d="M8 12l4-4 4 4M12 8v8"/>
            </svg>
            <p><strong>Click to upload</strong> or drag & drop</p>
            <p>PNG, JPG, JPEG, WEBP — <span id="multi-hint">one file</span></p>
          </div>
          <input type="file" id="file-input" accept="image/*" onchange="onFileChange(event)"/>
          <div class="preview-grid" id="preview-grid"></div>
          <div class="controls">
            <div class="control-row">
              <label class="toggle">
                <input type="checkbox" id="auto-crop" checked/>
                Auto-crop document
              </label>
            </div>
            <div class="control-row">
              <button class="btn btn-primary" id="run-btn" onclick="runOCR()" disabled>🔍 Run OCR</button>
              <button class="btn btn-ghost" onclick="clearFiles()">Clear</button>
            </div>
          </div>
          <div class="status" id="img-status"></div>
          <progress id="img-progress" style="display:none" max="100"></progress>
        </div>

        <div class="card">
          <div class="card-title" style="justify-content:space-between">
            <span>📄 Extracted Text</span>
            <button class="copy-btn" onclick="copyOutput()">Copy</button>
          </div>
          <div id="single-output">
            <div class="output-box empty" id="output-text">Extracted text will appear here…</div>
            <div class="output-meta" id="output-meta"></div>
          </div>
          <div id="batch-output" style="display:none">
            <div id="batch-results"></div>
          </div>
        </div>
      </div>
    </div>

    <!-- ═══ CAMERA SECTION ═══ -->
    <div class="section" id="sec-camera">
      <div class="card">
        <div class="cam-controls">
          <button class="btn btn-primary" id="cam-start-btn" onclick="startCamera()">📷 Start Camera</button>
          <button class="btn btn-danger" id="cam-stop-btn" onclick="stopCamera()" style="display:none">⏹ Stop</button>
          <span class="live-badge" id="live-badge" style="display:none">
            <span class="live-dot"></span> LIVE OCR
          </span>
          <span class="frame-info" id="frame-info"></span>
        </div>

        <div class="cam-layout">
          <div>
            <div class="card-title">📷 Camera Feed</div>
            <video id="camera-video" autoplay muted playsinline></video>
            <canvas id="camera-canvas"></canvas>
          </div>
          <div>
            <div class="card-title" style="justify-content:space-between">
              <span>📝 Live OCR Output</span>
              <button class="copy-btn" onclick="copyCameraOutput()">Copy</button>
            </div>
            <div class="output-box" id="camera-output"
                 style="min-height:280px;color:var(--muted);font-style:italic;font-family:inherit">
              Start the camera to begin live text extraction…
            </div>
            <div class="output-meta" id="camera-meta"></div>
          </div>
        </div>

        <div class="status" id="cam-status" style="margin-top:14px"></div>
      </div>
    </div>
  </main>
</div>
<div id="toast">Copied!</div>

<script>
const API = window.location.origin;
let imageMode='single', files=[];
let cameraStream=null, cameraWS=null, cameraTimer=null;
let frameCount=0, lastText='', serverBusy=false;
let elapsedInterval=null, elapsedStart=0;

// ── Tabs ──────────────────────────────────────────────────────────────
function switchTab(tab){
  document.querySelectorAll('.tab').forEach(t=>t.classList.toggle('active',t.dataset.tab===tab));
  document.querySelectorAll('.section').forEach(s=>s.classList.remove('active'));
  document.getElementById('sec-'+tab).classList.add('active');
}

// ── Image mode ────────────────────────────────────────────────────────
function setImageMode(mode){
  imageMode=mode;
  document.getElementById('pill-single').classList.toggle('active',mode==='single');
  document.getElementById('pill-batch').classList.toggle('active',mode==='batch');
  document.getElementById('file-input').multiple=mode==='batch';
  document.getElementById('multi-hint').textContent=mode==='batch'?'multiple files':'one file';
  clearFiles();
}

// ── Drag & Drop ───────────────────────────────────────────────────────
function onDragOver(e){e.preventDefault();document.getElementById('dropzone').classList.add('drag-over')}
function onDragLeave(e){document.getElementById('dropzone').classList.remove('drag-over')}
function onDrop(e){
  e.preventDefault();document.getElementById('dropzone').classList.remove('drag-over');
  const dropped=Array.from(e.dataTransfer.files).filter(f=>f.type.startsWith('image/'));
  imageMode==='single'?addFiles([dropped[0]].filter(Boolean)):addFiles(dropped);
}
function onFileChange(e){
  const sel=Array.from(e.target.files);
  imageMode==='single'?addFiles([sel[0]].filter(Boolean)):addFiles(sel);
  e.target.value='';
}

function addFiles(newFiles){
  files=imageMode==='single'?newFiles.slice(0,1):[...files,...newFiles];
  renderPreviews();
  document.getElementById('run-btn').disabled=files.length===0;
}
function removeFile(idx){files.splice(idx,1);renderPreviews();document.getElementById('run-btn').disabled=files.length===0;}
function clearFiles(){
  files=[];renderPreviews();
  document.getElementById('run-btn').disabled=true;
  resetOutput();
}
function renderPreviews(){
  const g=document.getElementById('preview-grid');g.innerHTML='';
  files.forEach((f,i)=>{
    const url=URL.createObjectURL(f);
    g.innerHTML+=`<div class="preview-item"><img src="${url}"/><button class="remove" onclick="removeFile(${i})">✕</button></div>`;
  });
}

// ── Elapsed counter ───────────────────────────────────────────────────
function startElapsed(statusId){
  stopElapsed();
  elapsedStart=Date.now();
  elapsedInterval=setInterval(()=>{
    const sec=((Date.now()-elapsedStart)/1000).toFixed(0);
    const el=document.getElementById(statusId);
    if(el){
      const txt=el.querySelector('.elapsed-txt');
      if(txt) txt.textContent=`${sec}s`;
    }
  },1000);
}
function stopElapsed(){if(elapsedInterval){clearInterval(elapsedInterval);elapsedInterval=null;}}

// ── OCR ───────────────────────────────────────────────────────────────
async function runOCR(){
  if(!files.length) return;
  const autoCrop=document.getElementById('auto-crop').checked;
  document.getElementById('run-btn').disabled=true;
  document.getElementById('img-progress').style.display='';
  animateProgress();
  try{
    if(imageMode==='single'||files.length===1) await runSingleOCR(files[0],autoCrop);
    else await runBatchOCR(files,autoCrop);
  }catch(err){
    setStatus('img-status',false,'❌ '+err.message);
  }finally{
    document.getElementById('run-btn').disabled=false;
    document.getElementById('img-progress').style.display='none';
    stopElapsed();
  }
}

let progressAnim=null;
function animateProgress(){
  const p=document.getElementById('img-progress');
  let v=0; p.value=0;
  if(progressAnim) clearInterval(progressAnim);
  progressAnim=setInterval(()=>{v=Math.min(v+0.5,90);p.value=v;},300);
}
function finishProgress(){
  if(progressAnim){clearInterval(progressAnim);progressAnim=null;}
  const p=document.getElementById('img-progress'); if(p) p.value=100;
}

async function runSingleOCR(file,autoCrop){
  const fd=new FormData(); fd.append('file',file); fd.append('auto_crop',autoCrop);
  setStatus('img-status',true,`Processing <b>${file.name}</b>… <span class="elapsed-txt">0s</span>`);
  startElapsed('img-status');
  const res=await fetch(`${API}/ocr/image`,{method:'POST',body:fd});
  const data=await res.json();
  finishProgress();
  stopElapsed();
  if(data.error) throw new Error(data.error);
  document.getElementById('single-output').style.display='';
  document.getElementById('batch-output').style.display='none';
  const box=document.getElementById('output-text');
  box.className='output-box'; box.style.color='var(--accent2)';
  box.textContent=data.text;
  document.getElementById('output-meta').innerHTML=
    `<div class="meta-chip">⏱ <span>${data.elapsed}s</span></div>`+
    `<div class="meta-chip">📊 Score <span>${data.score}</span></div>`+
    `<div class="meta-chip">🖥 <span>${data.device||'?'}</span></div>`+
    (data.angle_corrected?`<div class="meta-chip">🔄 <span>${data.angle_corrected}°</span></div>`:'');
  setStatus('img-status',false,`✅ Done in ${data.elapsed}s`);
}

async function runBatchOCR(fileList,autoCrop){
  const fd=new FormData();
  fileList.forEach(f=>fd.append('files',f)); fd.append('auto_crop',autoCrop);
  setStatus('img-status',true,`Processing ${fileList.length} images… <span class="elapsed-txt">0s</span>`);
  startElapsed('img-status');
  const res=await fetch(`${API}/ocr/batch`,{method:'POST',body:fd});
  const data=await res.json();
  finishProgress(); stopElapsed();
  document.getElementById('single-output').style.display='none';
  document.getElementById('batch-output').style.display='';
  const container=document.getElementById('batch-results');
  container.innerHTML='';
  data.results.forEach(r=>{
    container.innerHTML+=`
      <div class="batch-result">
        <div class="batch-filename">
          <span>📄 ${r.filename}</span>
          <div style="display:flex;gap:6px;align-items:center">
            <span class="meta-chip">⏱ <span>${r.elapsed}s</span></span>
            <button class="copy-btn" onclick="copyText(${JSON.stringify(r.text)},this)">Copy</button>
          </div>
        </div>
        <div class="batch-text">${esc(r.text)}</div>
      </div>`;
  });
  setStatus('img-status',false,`✅ ${data.count} images processed`);
}

function resetOutput(){
  document.getElementById('single-output').style.display='';
  document.getElementById('batch-output').style.display='none';
  const box=document.getElementById('output-text');
  box.className='output-box empty'; box.textContent='Extracted text will appear here…';
  document.getElementById('output-meta').innerHTML='';
}

// ── Camera ────────────────────────────────────────────────────────────
async function startCamera(){
  try{
    cameraStream=await navigator.mediaDevices.getUserMedia({video:{width:{ideal:640},height:{ideal:480}}});
    const video=document.getElementById('camera-video');
    video.srcObject=cameraStream; await video.play();
    document.getElementById('cam-start-btn').style.display='none';
    document.getElementById('cam-stop-btn').style.display='';
    document.getElementById('live-badge').style.display='';
    setStatus('cam-status',true,'Connecting to OCR server…');
    connectCameraWS();
  }catch(err){setStatus('cam-status',false,'❌ Camera error: '+err.message);}
}

function connectCameraWS(){
  const wsURL=`${API.replace('http','ws')}/ocr/camera`;
  cameraWS=new WebSocket(wsURL);
  cameraWS.onopen=()=>{
    setStatus('cam-status',false,'✅ Connected — sending frames every 5s');
    startFrameLoop();
  };
  cameraWS.onmessage=(e)=>{
    const data=JSON.parse(e.data);
    if(data.heartbeat) return;
    if(data.error){setStatus('cam-status',false,'⚠️ '+data.error);return;}

    const box=document.getElementById('camera-output');

    if(data.busy){
      box.className='output-box busy';
      box.style.fontStyle='italic'; box.style.fontFamily='inherit';
      box.textContent='⏳ Server processing previous frame…';
      document.getElementById('frame-info').textContent=`Frame #${data.frame_id} — server busy`;
      return;
    }

    const isNew=data.text!==lastText && data.text && !data.text.includes('[No text');
    box.className='output-box';
    box.style.fontFamily='Courier New,monospace'; box.style.fontStyle='normal';
    box.style.color=isNew?'var(--accent2)':'var(--muted)';
    box.textContent=data.text; lastText=data.text;
    document.getElementById('camera-meta').innerHTML=
      `<div class="meta-chip">⏱ <span>${data.elapsed}s</span></div>`+
      `<div class="meta-chip">Frame <span>#${data.frame_id}</span></div>`+
      `<div class="meta-chip">${isNew?'🟢 New text':'🔵 Same'}</div>`+
      `<div class="meta-chip">🖥 <span>${data.device||'?'}</span></div>`;
    document.getElementById('frame-info').textContent=
      `Frames sent: ${frameCount} | Last: ${data.elapsed}s`;
    serverBusy=false;
    setStatus('cam-status',false,'✅ Connected — sending frames every 5s');
  };
  cameraWS.onerror=()=>setStatus('cam-status',false,'❌ WebSocket error');
  cameraWS.onclose=()=>{setStatus('cam-status',false,'WebSocket closed');stopFrameLoop();serverBusy=false;};
}

function startFrameLoop(){stopFrameLoop();cameraTimer=setInterval(sendFrame,5000);sendFrame();}
function stopFrameLoop(){if(cameraTimer){clearInterval(cameraTimer);cameraTimer=null;}}

function sendFrame(){
  if(!cameraWS||cameraWS.readyState!==WebSocket.OPEN) return;
  const video=document.getElementById('camera-video');
  const canvas=document.getElementById('camera-canvas');
  if(video.readyState<2||video.videoWidth===0) return;
  if(serverBusy){
    document.getElementById('frame-info').textContent=`Frames sent: ${frameCount} | Server busy, skipping…`;
    return;
  }
  canvas.width=video.videoWidth; canvas.height=video.videoHeight;
  canvas.getContext('2d').drawImage(video,0,0);
  frameCount++;
  serverBusy=true;
  setStatus('cam-status',true,'Processing frame…');
  cameraWS.send(JSON.stringify({frame:canvas.toDataURL('image/jpeg',0.85)}));
}

function stopCamera(){
  stopFrameLoop();
  if(cameraWS){cameraWS.close();cameraWS=null;}
  if(cameraStream){cameraStream.getTracks().forEach(t=>t.stop());cameraStream=null;}
  document.getElementById('camera-video').srcObject=null;
  document.getElementById('cam-start-btn').style.display='';
  document.getElementById('cam-stop-btn').style.display='none';
  document.getElementById('live-badge').style.display='none';
  frameCount=0; serverBusy=false;
  document.getElementById('frame-info').textContent='';
  setStatus('cam-status',false,'📷 Camera stopped');
}

// ── Utilities ─────────────────────────────────────────────────────────
function setStatus(id,loading,msg){
  document.getElementById(id).innerHTML=
    msg?`${loading?'<div class="spinner"></div>':''}<span>${msg}</span>`:'';
}
function esc(s){return(s||'').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');}
function copyOutput(){copyText(document.getElementById('output-text').textContent);}
function copyCameraOutput(){copyText(document.getElementById('camera-output').textContent);}
function copyText(text,btn){
  navigator.clipboard.writeText(text).then(()=>{
    showToast('Copied!');
    if(btn){btn.classList.add('copied');btn.textContent='Copied!';
      setTimeout(()=>{btn.classList.remove('copied');btn.textContent='Copy';},2000);}
  });
}
function showToast(msg){
  const t=document.getElementById('toast');t.textContent=msg;t.classList.add('show');
  setTimeout(()=>t.classList.remove('show'),2000);
}

// ── Health check ──────────────────────────────────────────────────────
(async function checkHealth(){
  try{
    const r=await fetch(`${API}/health`); const d=await r.json();
    const b=document.getElementById('gpu-badge');
    const isCPU=d.gpu.includes('CPU');
    b.textContent=(isCPU?'⚠️ ':'✅ ')+d.gpu;
    b.className='badge'+(isCPU?' warn':'');
    if(isCPU) document.getElementById('cpu-warn').classList.add('show');
  }catch{
    const b=document.getElementById('gpu-badge');
    b.textContent='⚠️ Server offline'; b.className='badge err';
  }
})();
</script>
</body>
</html>'''

pathlib.Path("static").mkdir(exist_ok=True)
pathlib.Path("static/index.html").write_text(html_code.strip())
print("✅ static/index.html written")
print("✅ All files ready — run Cell 5")

✅ static/index.html written
✅ All files ready — run Cell 5


In [ ]:
import subprocess, threading, time, os, urllib.request, stat, re
import nest_asyncio, asyncio, uvicorn
nest_asyncio.apply()

PORT = 8000

CF_BIN = '/tmp/cloudflared'
if not os.path.exists(CF_BIN):
    print('⬇️  Downloading cloudflared...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CF_BIN)
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)
    print('✅ cloudflared downloaded')
else:
    print('✅ cloudflared already present')

cf_log = '/tmp/cf_tunnel.log'
open(cf_log, 'w').close()
subprocess.Popen([CF_BIN, 'tunnel', '--url', f'http://localhost:{PORT}'],
                 stdout=open(cf_log,'w'), stderr=subprocess.STDOUT)

# Watch log — print URL only ONCE
def watch_log():
    printed = False
    while True:
        try:
            urls = re.findall(r'https://[a-z0-9\-]+\.trycloudflare\.com', open(cf_log).read())
            if urls and not printed:
                printed = True
                print('\n' + '═'*62)
                print(f'  🔗 PUBLIC URL: {urls[0]}')
                print('  Open this URL in your browser!')
                print('═'*62 + '\n')
        except: pass
        time.sleep(1)

threading.Thread(target=watch_log, daemon=True).start()
print(f'🌐 Cloudflare Tunnel starting... waiting for URL (10–20s)\n')
time.sleep(4)

os.environ['CLOUDFLARE_TUNNEL'] = '0'
print('🚀 Starting OCR API server...')
config = uvicorn.Config('main:app', host='0.0.0.0', port=PORT, log_level='warning')
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

⬇️  Downloading cloudflared...
✅ cloudflared downloaded
🌐 Cloudflare Tunnel starting... waiting for URL (10–20s)

🚀 Starting OCR API server...

══════════════════════════════════════════════════════════════
  🔗 PUBLIC URL: https://stuff-hamilton-prove-finance.trycloudflare.com
  Open this URL in your browser!
══════════════════════════════════════════════════════════════

✅ GPU detected: Tesla T4
🔄 Loading Qwen2-VL-2B-Instruct…


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Qwen2-VL ready on cuda:0!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


📷 Camera WS disconnected
📷 Camera WS disconnected
📷 Camera WS disconnected
